<a href="https://colab.research.google.com/github/Hira-Tech-GenAi/learn-agentic-ai_projects/blob/main/langchain_rag_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time

index_name = "rag-with-langchain"  # change if desired

pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )


index = pc.Index(index_name)

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os


os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")



embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")


In [ ]:
vector = embeddings.embed_query("Hello! My name is Hira Khalid.")
print(vector[:5])

[0.020371248945593834, 0.0033172753173857927, -0.06449943780899048, -0.031583480536937714, 0.03906623646616936]


In [ ]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [ ]:
#data save

from uuid import uuid4

from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocalate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

#data retrieve

In [ ]:
len(documents)

10

In [ ]:
#Asign unique id
uuids = [str(uuid4()) for _ in range(len(documents))]

vector_store.add_documents(documents=documents, ids=uuids)

['d742e46e-73e4-47a2-ad9e-d7558fbfac2d',
 'baf6f8f6-2c83-48ca-b1a8-02fb01e7c0eb',
 '0efdf93c-c9c3-4815-9b56-98b333354f02',
 '03a4fcca-f550-4f15-9f0b-1482f0f81d35',
 'd19968b4-7552-4434-8b1a-a5a984f9702f',
 '0614c330-fe10-427f-b0bf-cec0920f06ae',
 'f1a8ad4b-0cae-4601-950e-df7e28be89e6',
 'b2312645-4200-4f78-8cd6-ccd74f4dd012',
 'cdb54ddb-5c23-4b78-9188-81c08e3d06e4',
 'de7dec43-24b4-4b62-99eb-49b692973f91']

In [ ]:
#Delete items from from vector store
#'vector_store.delete(ids=[uuids[-1]])'
#=====================================#

#Performing a simple similarity search can be done as follows:

#Retrieve Data
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
       k=2,
    filter={"source": "tweet"},


)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")


* LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]
* Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]


In [ ]:
#Similarity search with score

results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?", k=1, filter={"source": "news"}
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")



* [SIM=0.667716] The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. [{'source': 'news'}]


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)

In [ ]:


def answer_to_user_question(query: str):

  #vactor search
  vector_results = vector_store.similarity_search(query, k=2)

  #Pass to Model vector results + User Query
  final_answer =llm.invoke(f"""Answer this Query: {query}, Here are some refrence to the answer {vector_results}""")
  return final_answer


In [ ]:
answer = answer_to_user_question("LangChain provides abstractions to make working with LLMs easy")


In [ ]:
answer.content

'The provided text mentions LangChain in the context of building a project, implying it simplifies the process.  However, neither document directly supports the statement that LangChain provides abstractions to make working with LLMs easy.  The documents only show LangChain is being used for a project and that another framework (LangGraph) exists for building certain types of applications.  More information is needed to confirm the original statement.\n'